# 13 — Addenda ulasan WACV (CPU saja, tanpa GPU)

Tiga hal yang diminta ulasan, semuanya di dump yang sudah ada. **Tidak ada GPU, tidak ada
citra**, hanya dump skor dan bobot kepala.

| fase | pertanyaan ulasan | run | perkiraan |
|---|---|---|---|
| **R** | Q1: apakah premis Proposisi 1 mahal? `--frac-recal` memisahkan baris yang memberi `c` dari baris yang memberi `g_θ` dan `λ`. Dua fraksi, supaya tidak ada dugaan fraksinya disetel | 30 | ~65 mnt |
| **K** | Q6: INTERP-Q milik Ding et al. masuk tabel pesaing (ia **terdefinisi** di `n_y=0`) | 3 | ~25 mnt |
| **P** | Q6: PAS milik Ding et al. sebagai sumbu skor | 5 | ~12 mnt |
| **D** | Q3: cosine vs Euclidean, bias dibuang, jumlah tetangga lain | 20 | ~45 mnt |
| **M** | apakah kesimpulannya bergantung pada anggaran baris `max_rows=250_000`, yang asalnya anggaran memori? | 15 | ~65 mnt |

Total sekitar **3,5 jam CPU**. Setiap fase berdiri sendiri, jadi **boleh dijalankan satu
fase per sesi** — dan itu cara paling aman kalau RAM-nya mepet.

**Resume sudah diuji, bukan diklaim.** `pcc/tests/test_notebook_resume.py` mengambil sel
runner dari berkas `.ipynb` ini dan menjalankannya dengan driver tiruan yang menghitung
berapa kali sebuah konfigurasi benar-benar dihitung. Tesnya menemukan satu bug nyata: JSON
tidak punya tuple, jadi `drop_features=()` kembali sebagai `[]` dan pembandingnya menandai
**setiap** run sebagai basi — resume akan menghitung ulang seluruh notebook. Sudah
diperbaiki, dan tesnya menjaga agar tidak kembali. Kalau sesi mati, jalankan ulang semua sel
dan yang sudah selesai akan dibaca dari cache Drive.

Sel konfigurasi melakukan **pra-terbang memori** dan gagal dalam hitungan detik kalau ada
konfigurasi yang tidak akan muat, dan `build()` menolak run di dump primer yang tidak
melewatkan `max_rows` eksplisit. Keduanya ada karena sesi crash 2026-08-19: tiga fase lupa
melewatkan `max_rows`, memuat dump penuh 4,61 GB, dan yang rusak bukan cuma RAM-nya —
angkanya jadi tidak sebanding dengan tabel mana pun di paper.

Yang **tidak** ada di sini: TACP/sTACP (Liu dkk., AAAI 2026). Penulisnya tidak melepas kode
dan mengevaluasi di CIFAR100-LT/ImageNet-LT yang tidak kita punya, jadi menurut §7 ia tidak
boleh masuk tabel; ia dibahas di related work saja.

## 1. Config repo, unduhan, dan Drive

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

# SUMBER DUMP. LTC sudah punya ID gdown konkret (dari notebook 00) -> nol langkah
# manual. CCC belum: ID-nya harus dibaca dari download_data.sh mereka. Jadi survei
# LTC dulu; pindah ke CCC hanya kalau tidak ada dump LTC yang memenuhi premis.
# TIDAK ADA saklar sumber. Versi sebelumnya punya SOURCE yang harus diedit manual,
# dan default-nya ('ltc') sudah diketahui GAGAL premis — jadi setiap run default
# berakhir dengan assert. Sekarang SEMUA sumber disurvei dalam satu jalan dan yang
# terbaik dipakai. Tabel perbandingannya sendiri adalah temuan yang dicari.
CCC_DATASETS = ('imagenet',)   # tambah 'inaturalist' (iNat-2021, 633 kelas) bila perlu
GID_SCORES_LTC = {'plantnet': '1k_PPQV3VJT44hz02CcnbqPstjQo70vGr',
                  'inaturalist': '1W8R8Jj2bhS2PbR-3X9vEw-WkanbOk6mq'}
# ID CCC dari download_data.sh mereka. Ditanam supaya TIDAK ada langkah manual:
# versi sebelumnya hanya mencetak skripnya dan menyuruh menjalankan gdown sendiri,
# yang membingungkan dan tidak perlu begitu ID-nya diketahui.
GID_SCORES_CCC = {'imagenet':   '1AQjUn3m010N_i6-sfD690W7mZq2RTwJz',   # 4,62 GB
                  'inaturalist':'1BUlQZhS_5x2LJpyxCGI1IkmRrkvmRD88',   # 6,72 GB (iNat-2021, 633 kelas)
                  'places365':  '119k7PE1l72fg5Rpez5brIOn28BwClqv2',   # 0,54 GB (365 kelas)
                  'cifar-100':  '1yXD9XqBxEJnJxcfnnduNK6nHHUU3iX_6'}   # 0,01 GB (100 kelas)
# Catatan daya uji: premis butuh >=500 kelas layak, jadi places365 (365 kelas) dan
# cifar-100 (100 kelas) TIDAK BISA memenuhinya secara konstruksi, berapa pun
# sampel per kelasnya. Yang mungkin: imagenet (1.000) dan iNat-2021 (633).
LTC_DATASETS = ('inaturalist', 'plantnet')   # rilis memuat varian -trunc juga
LOSS_VARIANT = 'cross_entropy'  # 'cross_entropy' | 'focal'. LTC mengirim SEMBILAN
                                # berkas dengan NAMA IDENTIK di subdirektori berbeda;
                                # tercampur = skor dari model lain, akurasi mirip,
                                # kalibrasi beda total. Notebook 00 kena isu yang sama.
# places365 (365 kelas) dan cifar-100 (100 kelas) TIDAK BISA memenuhi premis >=500
# kelas secara konstruksi, berapa pun sampel per kelasnya — jadi tidak diunduh.
N_CLASSES_EXPECTED = None      # None = jangan dipaksakan; dibaca dari dump

# --- PRIMER, ditetapkan di prereg_imagenet_gate.md. JANGAN diubah setelah melihat hasil.
ALPHA_PRIMARY = 0.10
N_CAL_PRIMARY = 25
N_BOOT_CLASS  = 400           # bootstrap tingkat-kelas untuk gate B
N_PERM_CLASS  = 1000          # permutasi tingkat-kelas untuk gate C (p_min = 1/1001)
STABLE_THRESHOLD = 0.90

# --- SEKUNDER
ALPHAS_SECONDARY = (0.01, 0.05)
N_CAL_SECONDARY  = 50
N_SPLITS_BC = 100
N_SPLITS_A  = 100
RUN_CLUSTERED_CP = True       # reproduksi baseline pada skor yang sama

FRAC_DESC, FRAC_CAL = 0.40, 0.30   # sisanya EVAL

# ANGGARAN BARIS. Dump ImageNet CCC nyata adalah (1.153.051 x 1.000) float32 = 4,61 GB
# -- sepuluh kali lebih besar dari yang tercatat di release_audit.md. Memuatnya penuh
# lalu membuat salinan turunan (thr_lac, entropi, np.partition) melewati RAM Colab.
#
# Subsampling di sini BUKAN perubahan kriteria: premis butuh >=84 sampel/kelas dan
# anggaran ini menyisakan ~230/kelas. Ia diambil sebagai FRAKSI per kelas, bukan cap
# tetap, karena cap tetap membuat semua hitungan kelas SAMA -> log_prevalence konstan
# -> ablasi prevalensi jadi hampa. Fraksi mempertahankan struktur prevalensinya.
MAX_ROWS = 250_000            # None = pakai seluruh dump
SEED = 42
# =======================================================================
print(f'PRIMER: alpha={ALPHA_PRIMARY} n_cal={N_CAL_PRIMARY} '
      f'n_boot={N_BOOT_CLASS} n_perm={N_PERM_CLASS}')


## 2. Mount + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
subprocess.run(['pip','install','-q','gdown'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Siapkan SEMUA dump — TANPA citra, TANPA GPU

Dump LTC dipakai lokasi yang sama dengan notebook 00 (`released_scores/<dataset>`), jadi kalau
sudah ada tidak diunduh ulang. Dump CCC diunduh otomatis dengan ID yang sudah ditanam.

Keduanya disurvei berdampingan di sel 4. Itu bukan pemborosan: **perbandingan ekor-panjang
versus berimbang adalah temuannya**, dan menurunkannya dari satu tabel lebih kuat daripada dari
dua run terpisah.


In [ ]:
import glob, zipfile, numpy as np

def ltc_dir(ds):
    return f'{DRIVE_ROOT}/released_scores/{ds}'

def ccc_dir(ds):
    return f'{DRIVE_ROOT}/scores_ccc/{ds}'

for ds in LTC_DATASETS:
    d = ltc_dir(ds); os.makedirs(d, exist_ok=True)
    if glob.glob(f'{d}/**/*_softmax.npy', recursive=True):
        print(f'ltc/{ds}: sudah ada, dilewati')
        continue
    print(f'ltc/{ds}: mengunduh...')
    z = f'{d}/{ds}.zip'
    r = subprocess.run(['gdown', GID_SCORES_LTC[ds], '-O', z], capture_output=True, text=True)
    if r.returncode:
        print('  gdown gagal:', r.stderr.strip()[:300])
    else:
        subprocess.run(['unzip','-o','-q',z,'-d',d], check=False)

def inventory(d, label):
    files = [p for p in sorted(glob.glob(f'{d}/**/*', recursive=True)) if os.path.isfile(p)]
    print(f'  isi {label}: {len(files)} berkas')
    for p in files[:25]:
        print(f'    {os.path.relpath(p, d):56s} {os.path.getsize(p)/1e6:9.2f} MB')
    return files

def looks_like_html(p):
    # Kegagalan kuota Google Drive menulis halaman HTML DAN mengembalikan kode 0.
    # Inilah sebabnya returncode tidak boleh dipercaya sebagai bukti unduhan berhasil.
    try:
        with open(p, 'rb') as fh:
            head = fh.read(400)
    except OSError:
        return False, b''
    low = head.lower()
    return (b'<html' in low or b'<!doctype html' in low), head

for ds in CCC_DATASETS:
    d = ccc_dir(ds); os.makedirs(d, exist_ok=True)
    mat = f'/content/ccc_npy/{ds}'
    # Tiga keadaan, dan versi sebelumnya hanya mengenali yang pertama:
    #   (a) .npy sudah dimaterialkan di /content -> tidak ada kerja
    #   (b) .npz ada di Drive tapi .npy hilang (sesi baru; /content ephemeral)
    #       -> ekstrak ulang, JANGAN unduh 4,6 GB lagi
    #   (c) tidak ada apa pun -> unduh
    # Pemeriksaan lama hanya mencari .npy DI DRIVE, yang tidak pernah ada karena
    # materialisasinya ke /content. Jadi setiap sesi baru mengunduh ulang 4,6 GB.
    if glob.glob(f'{mat}/*.npy') or glob.glob(f'{d}/**/*.npy', recursive=True):
        print(f'ccc/{ds}: .npy sudah ada, dilewati')
        continue
    if glob.glob(f'{d}/*.npz'):
        print(f'ccc/{ds}: .npz ada di Drive, ekstrak ulang tanpa mengunduh')
    else:
        print(f'ccc/{ds}: mengunduh (beberapa GB, sabar)...')
    if not glob.glob(f'{d}/*.npz'):
        r = subprocess.run(['gdown', '--fuzzy', GID_SCORES_CCC[ds]],
                           cwd=d, capture_output=True, text=True)
        if r.stdout.strip():
            print('  stdout:', r.stdout.strip()[-500:])
        if r.stderr.strip():
            print('  stderr:', r.stderr.strip()[-300:])
        print(f'  returncode: {r.returncode}  <- BUKAN bukti; diverifikasi di bawah')

    for p in sorted(glob.glob(f'{d}/*')):
        low = p.lower()
        if low.endswith('.zip'):
            subprocess.run(['unzip','-o','-q',p,'-d',d], check=False)
        elif low.endswith(('.tar.gz','.tgz','.tar')):
            subprocess.run(['tar','-xf',p,'-C',d], check=False)

    # .npz ADALAH zip berisi beberapa .npy. Versi sebelumnya mencocokkan ekstensi
    # secara literal ('.zip'/'.tar'), jadi imagenet.npz dilewati dan tidak ada .npy
    # terbentuk -- padahal unduhannya berhasil penuh. Satu baris, kegagalan bisu.
    #
    # Header .npy dibaca lewat zipfile TANPA mendekompresi isinya, supaya bentuk dan
    # dtype tiap anggota terlihat tanpa memuat gigabyte. Lalu HANYA dua array yang
    # dibutuhkan dimaterialkan, dan ke /content (ephemeral) bukan Drive -- mengekstrak
    # seluruh 4,6 GB ke Drive akan menggandakan pemakaian kuota tanpa alasan.
    for p in sorted(glob.glob(f'{d}/*.npz')):
        print(f'  membaca header {os.path.basename(p)} (tanpa dekompresi)...')
        members = []
        with zipfile.ZipFile(p) as zf:
            for nm in zf.namelist():
                try:
                    with zf.open(nm) as fh:
                        ver = np.lib.format.read_magic(fh)
                        if ver == (1, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_1_0(fh)
                        elif ver == (2, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_2_0(fh)
                        else:
                            continue
                    members.append((nm, shp, dt))
                    print(f'    {nm:36s} {str(shp):20s} {dt}')
                except Exception as e:
                    print(f'    {nm:36s} header tak terbaca: {type(e).__name__}')

        twod = [m for m in members if len(m[1]) == 2]
        oned = [m for m in members if len(m[1]) == 1]
        if not twod or not oned:
            print('  npz ini tidak memuat pasangan (2-D, 1-D) — kirim daftar di atas.')
            continue
        sc = max(twod, key=lambda m: m[1][0] * m[1][1])
        lb = next((m for m in oned if m[1][0] == sc[1][0]), None)
        if lb is None:
            print(f'  tidak ada array 1-D sepanjang {sc[1][0]} untuk mendampingi {sc[0]}')
            continue
        out = f'/content/ccc_npy/{ds}'
        os.makedirs(out, exist_ok=True)
        print(f'  materialkan {sc[0]} {sc[1]} dan {lb[0]} {lb[1]} -> {out}')
        # zipfile.namelist() memberi nama DENGAN sufiks '.npy', tetapi NpzFile
        # diindeks TANPA sufiks -> z['softmax.npy'] KeyError, z['softmax'] benar.
        # Ditemukan oleh tes sintetik sebelum run nyata.
        k_sc = sc[0][:-4] if sc[0].endswith('.npy') else sc[0]
        k_lb = lb[0][:-4] if lb[0].endswith('.npy') else lb[0]
        with np.load(p, allow_pickle=False) as z:
            np.save(f'{out}/scores.npy', z[k_sc])
            np.save(f'{out}/labels.npy', z[k_lb])

    files = inventory(d, f'ccc/{ds}')
    npys = (glob.glob(f'{d}/**/*.npy', recursive=True)
            + glob.glob(f'/content/ccc_npy/{ds}/*.npy'))
    if not npys:
        print(f'  GAGAL: tidak ada .npy terbentuk untuk ccc/{ds}.')
        for p in files:
            is_html, head = looks_like_html(p)
            if is_html:
                print(f'  PENYEBAB: {os.path.basename(p)} adalah HALAMAN HTML, bukan data.')
                print('  Itu batas kuota Google Drive; gdown tetap keluar dengan kode 0.')
                print('  cuplikan:', ' '.join(head.decode('utf-8','replace').split())[:300])
                print(f'  URL: https://drive.google.com/uc?id={GID_SCORES_CCC[ds]}')
                break
        else:
            print('  Berkas ADA tetapi tidak menghasilkan .npy — kirim daftar di atas.')
    else:
        print(f'  OK: {len(npys)} .npy siap dipakai')

# dua akar untuk CCC: Drive (kalau .npy langsung) dan /content (hasil materialisasi
# anggota .npz). Keduanya diperiksa supaya tidak peduli bentuk rilisnya.
roots = ([(ltc_dir(ds), 'ltc', ds) for ds in LTC_DATASETS]
         + [(ccc_dir(ds), 'ccc', ds) for ds in CCC_DATASETS]
         + [(f'/content/ccc_npy/{ds}', 'ccc', ds) for ds in CCC_DATASETS])
found = []
for rt, src, ds in roots:
    for f in sorted(glob.glob(f'{rt}/**/*.npy', recursive=True)):
        found.append((f, src, ds))
print()
print(f'total .npy: {len(found)}')
for f, src, ds in found[:60]:
    a = np.load(f, mmap_mode='r')
    print(f'  [{src}] {os.path.basename(f):52s} {str(a.shape):18s} {a.dtype}')
if not found:
    print('TIDAK ADA .npy sama sekali. Isi direktori mentah:')
    for rt, src, ds in roots:
        for p in sorted(glob.glob(f'{rt}/**/*', recursive=True))[:25]:
            if os.path.isfile(p):
                print(f'  {os.path.relpath(p, DRIVE_ROOT):64s} {os.path.getsize(p)/1e6:8.1f} MB')


### 3b. Referensi — dari mana ID CCC berasal (opsional, tidak perlu dijalankan)

ID di sel 1 diambil dari `download_data.sh` milik CCC. Sel ini hanya untuk memverifikasi
bahwa ID-nya belum berubah; ia **tidak diperlukan** untuk menjalankan notebook.


In [ ]:
SHOW_CCC_SCRIPT = False
if SHOW_CCC_SCRIPT:
    if not os.path.isdir('/content/ccc'):
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/tiffanyding/class-conditional-conformal.git',
                        '/content/ccc'], check=False)
    print(open('/content/ccc/download_data.sh').read())
    print('bandingkan dengan GID_SCORES_CCC di sel 1')
else:
    print('dilewati (ID sudah ditanam di sel 1)')


## 4. Dump, kepala, dan repo pesaing

In [ ]:
import glob

def _pair(scores):
    for suf in ('_softmax.npy', '_scores.npy', 'scores.npy'):
        if scores.endswith(suf):
            cand = scores[: -len(suf)] + suf.replace('softmax', 'labels').replace(
                'scores', 'labels')
            if os.path.exists(cand):
                return cand
    cand = os.path.join(os.path.dirname(scores), 'labels.npy')
    return cand if os.path.exists(cand) else None

DUMPS = {}

# CCC: materialisasi sel di atas menaruhnya di /content/ccc_npy/<ds>/
for p in sorted(glob.glob('/content/ccc_npy/*/scores.npy')):
    ds = os.path.basename(os.path.dirname(p))
    lab = _pair(p)
    if lab:
        DUMPS['ccc_' + ds] = {'scores': p, 'labels': lab, 'eval_scores': None,
                              'eval_labels': None, 'max_rows': MAX_ROWS}

# LTC: pasangkan cal (DESC+CAL) dengan test (EVAL penuh)
for ds in LTC_DATASETS:
    d = f'{DRIVE_ROOT}/released_scores/{ds}'
    cal = sorted(glob.glob(f'{d}/**/*cal_softmax.npy', recursive=True))
    tst = sorted(glob.glob(f'{d}/**/*test_softmax.npy', recursive=True))
    cal = [p for p in cal if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    tst = [p for p in tst if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    if cal and tst and _pair(cal[0]) and _pair(tst[0]):
        DUMPS['ltc_' + ds] = {'scores': cal[0], 'labels': _pair(cal[0]),
                              'eval_scores': tst[0], 'eval_labels': _pair(tst[0]),
                              'max_rows': None}

assert DUMPS, 'tidak ada dump siap pakai -- periksa sel penyiapan di atas'
for k, v in DUMPS.items():
    a = np.load(v['scores'], mmap_mode='r')
    line = '  ' + k.ljust(18) + ' cal ' + str(a.shape)
    if v['eval_scores']:
        e = np.load(v['eval_scores'], mmap_mode='r')
        line += '  eval ' + str(e.shape) + '  (dump terpisah)'
    else:
        line += '  eval = 30% dari dump yang sama'
    print(line)

In [ ]:
import numpy as np, os, glob, subprocess

HEAD_DIR = '/content/head'
os.makedirs(HEAD_DIR, exist_ok=True)
HEADS = {}          # K -> (path W, path b)

def _register(tag, W, b):
    w_p = f'{HEAD_DIR}/{tag}_fc_weight.npy'
    b_p = f'{HEAD_DIR}/{tag}_fc_bias.npy'
    np.save(w_p, W)
    np.save(b_p, np.zeros(len(W)) if b is None else b)
    HEADS[int(W.shape[0])] = (w_p, b_p)
    print('  kepala', tag, W.shape, '-> K =', W.shape[0])

# --- 1. torchvision ResNet-50 (ImageNet-1k). Model tersupervisi yang BERBEDA dari
# SimCLRv2+probe penghasil skor CCC: ketidakcocokan itu justru yang membuat phi
# eksogen sepenuhnya terhadap delta_y.
tv_w = f'{HEAD_DIR}/torchvision_imagenet_fc_weight.npy'
if os.path.exists(tv_w):
    W = np.load(tv_w)
    HEADS[int(W.shape[0])] = (tv_w, tv_w.replace('_weight', '_bias'))
    print('  kepala torchvision_imagenet sudah ada -> K =', W.shape[0])
else:
    from pcc.descriptors.head_weights import load_torchvision_resnet50_head
    W, b = load_torchvision_resnet50_head()
    _register('torchvision_imagenet', W, b)

# --- 2. Kepala LTC untuk Pl@ntNet dan iNat-2018. Run pertama Phase 2 gagal di kedua
# dataset itu, tetapi HANYA keluarga ruang-output yang pernah dijalankan di sana --
# dan keluarga itu juga gagal di ImageNet. Jadi itu kegagalan KELUARGA phi, bukan
# kegagalan dataset, dan checkpoint yang dirilis membuatnya bisa diuji tanpa GPU.
GID_MODELS = '1tS-M-4IYyCGMeIxxyrgx2-XCZgdvw18S'   # models.zip, dari notebook 00
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints/ltc_models'
os.makedirs(CKPT_DIR, exist_ok=True)
if not glob.glob(f'{CKPT_DIR}/**/*model*.pth', recursive=True):
    print('mengunduh models.zip LTC (6 ResNet-50)...')
    subprocess.run(['gdown', GID_MODELS, '-O', f'{CKPT_DIR}/models.zip'], check=True)
    subprocess.run(['unzip', '-o', f'{CKPT_DIR}/models.zip', '-d', CKPT_DIR],
                   check=True)

def _variant_ok(path):
    # PERANGKAP dari notebook 00: LTC mengirim ENAM model dengan NAMA BERKAS
    # IDENTIK dan menaruh varian focal di subdirektori 'focal_loss'. Glob rekursif
    # bisa mengambil mana saja, jadi checkpoint dan skor bisa diam-diam berasal dari
    # varian berbeda -- akurasi mirip, kalibrasi beda total.
    is_focal = 'focal' in path.replace(chr(92), '/').lower()
    return is_focal if LOSS_VARIANT == 'focal' else (not is_focal)

from pcc.data.ltc_datasets import NUM_CLASSES
for ds in LTC_DATASETS:
    tag = 'ltc_' + ds
    w_p = f'{HEAD_DIR}/{tag}_fc_weight.npy'
    if os.path.exists(w_p):
        W = np.load(w_p, mmap_mode='r')
        HEADS[int(W.shape[0])] = (w_p, w_p.replace('_weight', '_bias'))
        print('  kepala', tag, 'sudah ada -> K =', W.shape[0])
        continue
    cands = sorted(glob.glob(f'{CKPT_DIR}/**/best-{ds}-model.pth', recursive=True))
    keep = [p for p in cands if _variant_ok(p)]
    print('  checkpoint', ds, ':', len(cands), 'kandidat,', len(keep),
          'cocok varian', LOSS_VARIANT)
    for p in cands:
        print('     ' + ('* ' if _variant_ok(p) else '  ') + p)
    if not keep:
        print('     DILEWATI: tidak ada checkpoint varian', LOSS_VARIANT)
        continue
    try:
        from pcc.extract.backbones import load_ltc_resnet50
        m = load_ltc_resnet50(keep[0], NUM_CLASSES[ds], None)
        W = m.fc.weight.detach().cpu().numpy()
        b = m.fc.bias.detach().cpu().numpy() if m.fc.bias is not None else None
        del m
        assert W.shape[0] == NUM_CLASSES[ds], (W.shape, NUM_CLASSES[ds])
        _register(tag, W, b)
    except Exception as e:
        print('     GAGAL memuat:', type(e).__name__, str(e)[:160])

print()
print('kepala tersedia per jumlah kelas:', {k: os.path.basename(v[0])
                                            for k, v in sorted(HEADS.items())})

### 4b. Repo LTC — sumber pesaing terbit

Metodenya dipanggil dari rilis penulisnya, tidak ditulis ulang. `fuzzy_classwise_CP`
memakai `np.quantile(..., weights=)` yang baru ada di **numpy 2.0**, jadi versinya
diperiksa di sini — kalau tidak, kegagalannya muncul 25 menit kemudian sebagai
`TypeError` di tengah fase 3.

In [ ]:
LTC_URL = 'https://github.com/tiffanyding/long-tail-conformal.git'
LTC_DIR = '/content/ltc'
if not (os.path.isdir(LTC_DIR) and os.listdir(LTC_DIR)):
    subprocess.run(['git', 'clone', '--depth', '1', LTC_URL, LTC_DIR], check=True)
assert os.path.exists(LTC_DIR + '/utils/conformal_utils.py'), 'repo LTC tak lengkap'
subprocess.run(['pip', 'install', '-q', 'scikit-learn', 'matplotlib'], check=False)

import numpy as np
_v = tuple(int(x) for x in np.__version__.split('.')[:2])
print('numpy', np.__version__)
if _v < (2, 0):
    print('PERINGATAN: fuzzy_classwise_CP butuh np.quantile(weights=) dari numpy',
          '2.0. Fase 3 akan mencatat kegagalannya, bukan diam-diam melewatkannya.')
else:
    print('numpy cukup baru untuk fuzzy_classwise_CP')
print('repo pesaing:', LTC_DIR)

## 5. Penyimpan Drive

In [ ]:
import shutil, time
RUN_DIR = DRIVE_ROOT + '/runs/nb12_' + time.strftime('%Y%m%d_%H%M%S')
os.makedirs(RUN_DIR, exist_ok=True)
# CACHE_DIR sengaja TANPA stempel waktu: itu satu-satunya cara run berikutnya
# menemukan pekerjaan yang sudah selesai. RUN_DIR yang berstempel tetap ada
# sebagai snapshot per-run.
CACHE_DIR = DRIVE_ROOT + '/runs/nb12_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
_done = sorted(glob.glob(CACHE_DIR + '/nb12_*.json'))
print('tujuan:', RUN_DIR)
print('cache resume:', CACHE_DIR, '->', len(_done), 'run sudah selesai')

def save_to_drive(tag):
    n_ok = n_bad = 0
    for p in sorted(glob.glob('pcc/reports/*.json')):
        d = os.path.join(RUN_DIR, os.path.basename(p))
        try:
            if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(p):
                n_ok += 1
                continue
            shutil.copy2(p, d)
            ok = os.path.getsize(d) == os.path.getsize(p)
            n_ok += int(ok); n_bad += int(not ok)
        except Exception as e:
            n_bad += 1
            print('   gagal', os.path.basename(p), e)
    print('   [{}] tersimpan {} gagal {} -> Drive'.format(tag, n_ok, n_bad))
    return n_bad == 0

save_to_drive('awal')

## 6. Konfigurasi eksperimen — EDIT ME kedua

In [ ]:
# === EDIT ME ===========================================================
ALPHA         = 0.10
N_CAL_MAIN    = 25
FRAC_CAL_MAIN = 0.70                  # 177 baris cal/kelas, 76 untuk EVAL

# MAX_ROWS (didefinisikan di sel 2) WAJIB dilewatkan ke setiap run di dump primer, dan
# alasannya bukan memori. Setiap tabel nb12 yang akan dimasuki run di notebook ini memakai
# max_rows=250_000 dengan frac_cal=0.70, yaitu 177 baris kalibrasi dan 76 evaluasi per
# kelas. Melewatkan max_rows memuat dump PENUH (~1,15 juta baris) dan memberi ~524 baris
# kalibrasi per kelas -- dan blok kedalaman-kalibrasi di tab:depth menunjukkan 100->175
# baris/kelas saja sudah membeli +0.019 worst-class. Jadi baris PAS akan terlihat bagus
# karena kedalamannya, bukan karena PAS, dan lengan 'reuse' tidak akan mereproduksi
# ho0p3 = +0.0580. Kelalaian ini terjadi 2026-08-19 dan muncul sebagai sesi crash RAM;
# yang sebenarnya rusak adalah keterbandingannya. build() sekarang menolaknya.

# FASE R -- PCC-split. Sepuluh split, bukan lima: yang diklaim di sini adalah VALIDITAS,
# dan selisih coverage marginal yang kita cari besarnya ~0.002, jauh lebih kecil daripada
# efek worst-class. Kedua lengan dijalankan di notebook ini supaya berpasangan pada split
# yang sama; memakai angka nb12 sebagai lengan 'reuse' akan membandingkan split berbeda.
SEEDS_SPLIT   = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)
# DUA fraksi, bukan satu, supaya tidak ada pertanyaan apakah fraksinya disetel agar
# lengan split terlihat bagus. `c` adalah SATU skalar dari baris terkumpul, jadi ia butuh
# jumlah baris total, bukan kedalaman per kelas: pada 0.25 itu ~31k baris (slack konformal
# 3e-5) dan pada 0.50 ~62k (2e-5), keduanya puluhan kali lebih kecil daripada defisit
# 0.0024 yang diukur. Yang dibayar fraksi besar adalah kedalaman fit untuk g_theta.
FRAC_RECALS   = (0.25, 0.50)          # 0.25 -> 44 recal + 133 fit; 0.50 -> 88 + 89
ARMS_R        = [('reuse', 0.0)] + [('split%s' % str(f).replace('.', 'p'), f)
                                    for f in FRAC_RECALS]

# FASE K -- pesaing. Persis konfigurasi fase 3 nb12, supaya barisnya bisa ditempel ke
# tabel yang sama: 100k baris dengan frac_cal 0.50 memberi 50 cal + 50 eval per kelas,
# jadi regime A dengan margin dan statistik primernya sama dengan tabel lain.
SEEDS_COMP    = (0, 1, 2)
MAX_ROWS_COMP = 100_000
FRAC_CAL_COMP = 0.50

# FASE M -- anggaran baris. Setiap tabel di paper memakai max_rows=250_000, dan asalnya
# adalah anggaran memori: dump CCC penuh 1.153.051 x 1.000 float32 = 4,61 GB, dan
# memuatnya penuh lalu menyalin turunannya melewati RAM Colab. Subsample-nya SAH -- ia
# fraksi acak per kelas, jadi exchangeability utuh dan jaminan konformalnya tidak
# tersentuh -- dan kriteria pra-registrasi >=84 sampel/kelas terpenuhi dengan margin 2,7x.
#
# Tapi "sah" adalah argumen, bukan pengukuran. Fase ini mengubahnya jadi pengukuran, dan
# ia bisa GAGAL: kalau delta worst-class bergerak sistematis dengan anggaran baris, maka
# angka utama bergantung pada subsample dan itu harus dinyatakan. Prediksi kita, dari
# tab:depth dan fig:evaldepth, adalah efeknya NAIK dengan anggaran -- artinya 250k
# konservatif, bukan menguntungkan.
#
# 1.153.051 (dump penuh) TIDAK ada di sini: puncaknya ~9,2 GB dan itulah yang membuat
# sesi crash. Kalau kau pindah ke runtime dengan RAM lebih besar, tambahkan None ke tuple.
ROW_BUDGETS   = (250_000, 500_000, 750_000)

# FASE P dan D
SEEDS_MAIN    = (0, 1, 2, 3, 4)
DESC_ARMS = (
    ('euclidean',  dict(dist_metric='euclidean')),
    ('nobias',     dict(drop_features=('w_bias',))),
    ('knn_1_2_5',  dict(knn_ks=(1, 2, 5))),
    ('knn_1_10_20_100', dict(knn_ks=(1, 10, 20, 100))),
)
# =======================================================================

# Cache Drive SENDIRI. Nama laporan sudah berawalan nb13 jadi tidak akan bertumbukan
# dengan nb12, tapi mencampur 228 berkas nb12 dengan 48 berkas nb13 di satu folder
# membuat langkah "salin hasil terakhir" jadi tebak-tebakan. TIDAK berstempel waktu --
# itu yang membuat resume benar-benar menemukannya.
CACHE_DIR = DRIVE_ROOT + '/runs/nb13_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print('cache:', CACHE_DIR, '|', len(glob.glob(CACHE_DIR + '/*.json')), 'laporan tersimpan')

# VERIFIKASI VERSI. Notebook ini dan paket pcc/ harus sepasang, dan tidak ada apa pun di
# Colab yang menjamin itu: REPO_URL kosong, jadi repo masuk lewat cara lain dan bisa
# tertinggal beberapa commit. Notebook baru di atas paket lama gagal dengan pesan yang
# membingungkan; paket baru di bawah notebook lama berjalan diam-diam dengan konfigurasi
# yang salah. Dicek di sini supaya keduanya ketahuan dalam hitungan detik.
import inspect, subprocess
from pcc.experiments import phase2_pcc as _drv
from pcc.method.pcc import fit_pcc as _fit
from pcc.descriptors.head_weights import build_head_descriptors as _bhd
_need = [
    ('PCC-split (--frac-recal)',
     'score_matrix_recal' in inspect.signature(_fit).parameters),
    ('pesaing INTERP-Q', hasattr(_drv, 'INTERP_Q_WEIGHTS')),
    ('sumbu metrik deskriptor', 'metric' in inspect.signature(_bhd).parameters),
]
try:
    from pcc.scores.base import pas_scores as _pas
    _need.append(('skor PAS', True))
except ImportError:
    _need.append(('skor PAS', False))
_missing = [n for n, ok in _need if not ok]
assert not _missing, (
    'paket pcc/ TERTINGGAL dari notebook ini; yang hilang: %s. Jalankan `git pull` di %s, '
    'lalu Runtime > Restart session.' % (_missing, os.getcwd()))
try:
    _sha = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()
except Exception:
    _sha = ''
print('paket pcc/ lengkap:', ', '.join(n for n, _ in _need))
print('commit repo:', _sha or '(bukan checkout git)')

PRIMARY = 'ccc_imagenet'
assert PRIMARY in DUMPS, 'dump primer tak ada: ' + repr(sorted(DUMPS))
PRIMARY_S = DUMPS[PRIMARY]['scores']
PRIMARY_Y = DUMPS[PRIMARY]['labels']
_K = int(np.load(PRIMARY_S, mmap_mode='r').shape[1])
HEAD_W, HEAD_B = HEADS[_K]
print('dump primer', PRIMARY, '| K =', _K)
print('kepala:', os.path.basename(HEAD_W))
assert os.path.isdir(LTC_DIR), 'repo LTC wajib untuk fase K: ' + LTC_DIR

n_runs = (1 + len(FRAC_RECALS)) * len(SEEDS_SPLIT) + len(SEEDS_COMP) \
         + len(SEEDS_MAIN) + len(DESC_ARMS) * len(SEEDS_MAIN) \
         + len(ROW_BUDGETS) * len(SEEDS_MAIN)
print('total run:', n_runs)

# PRA-TERBANG MEMORI. Sesi crash 2026-08-19 karena tiga fase tidak melewatkan max_rows
# dan memuat dump penuh. Puncaknya diperkirakan di sini SEBELUM ada run yang jalan, jadi
# konfigurasi yang tidak akan muat gagal dalam hitungan detik, bukan setelah setengah jam.
def footprint(max_rows, frac_cal, frac_recal=0.0, K=_K):
    n = max_rows or 1_153_051
    gb = lambda r: r * K * 4 / 1e9
    cal = int(n * frac_cal * 0.7)
    ev = int(n * (1 - frac_cal))
    rec = int(cal * frac_recal)
    fit = cal - rec
    # S_all + fit + recal + eval, plus satu transient seukuran fit di dalam thr_lac
    return gb(n) + gb(fit) + gb(rec) + gb(ev) + gb(fit), fit // 700, ev // K

RAM_BUDGET_GB = 9.0        # sisakan ruang untuk kernel, torch, dan pesaing
print()
print('%-42s %8s %10s %8s' % ('konfigurasi', 'puncak', 'fit/kelas', 'eval/kelas'))
_plan = [('R reuse', MAX_ROWS, FRAC_CAL_MAIN, 0.0)] \
      + [('R split %.2f' % f, MAX_ROWS, FRAC_CAL_MAIN, f) for f in FRAC_RECALS] \
      + [('K pesaing', MAX_ROWS_COMP, FRAC_CAL_COMP, 0.0),
         ('P pas', MAX_ROWS, FRAC_CAL_MAIN, 0.0),
         ('D deskriptor', MAX_ROWS, FRAC_CAL_MAIN, 0.0)] \
      + [('M budget %s' % (b or 'PENUH'), b, FRAC_CAL_MAIN, 0.0) for b in ROW_BUDGETS]
_bad = []
for nm, mr, fc, fr in _plan:
    pk, f_pc, e_pc = footprint(mr, fc, fr)
    flag = '' if pk <= RAM_BUDGET_GB else '   <-- LEBIH DARI ANGGARAN'
    print('%-42s %6.2f GB %10d %8d%s' % (nm, pk, f_pc, e_pc, flag))
    if pk > RAM_BUDGET_GB:
        _bad.append((nm, pk))
assert not _bad, ('konfigurasi ini akan melewati RAM: %s. Turunkan anggaran barisnya atau '
                  'pindah ke runtime dengan RAM lebih besar.' % _bad)
print()
print('nb12 memakai max_rows=250_000 dengan frac_cal=0.70 -> 177 cal / 76 eval per kelas.')
print('Fase R, P, dan D HARUS memakai angka itu, karena barisnya masuk tabel yang sama.')

## 7. Runner bersama

`build()` memberi nilai pada **setiap** atribut, lalu penjaga `assert hasattr`
menolak nama yang tidak dikenal. Itu bukan hiasan: satu kali salah ketik nama
atribut pernah membunuh seluruh 48 konfigurasi notebook 08 karena diam-diam
dianggap default.

In [ ]:
import gc, json, traceback, collections, threading
try:
    import psutil
    _PROC = psutil.Process()
    def rss():
        return _PROC.memory_info().rss / 1e9
except Exception:
    def rss():
        return float('nan')

def mem(tag):
    print('   RSS {:5.2f} GB  {}'.format(rss(), tag), flush=True)
from pcc.experiments import phase2_pcc as drv
from pcc.utils.io import write_report
from pcc.eval.stats import mean_ci, holm_bonferroni

RESULTS, FAILED, PEAKS = [], [], []

def _sample_peak(stop, out, every=0.5):
    while not stop.is_set():
        v = rss()
        if v == v:                      # NaN when psutil is missing
            out[0] = max(out[0], v)
        stop.wait(every)

def peaks_report(top=12):
    """Which configurations came closest to the RAM ceiling."""
    if not PEAKS:
        print('belum ada run yang dihitung (semuanya dari cache?)')
        return
    rows = sorted(PEAKS, key=lambda d: -d['peak_gb'])
    print('%-6s %-22s %5s %9s %8s' % ('fase', 'tag', 'seed', 'puncak', 'detik'))
    for d in rows[:top]:
        print('%-6s %-22s %5d %7.2f GB %8.0f' % (
            d['phase'], d['tag'], d['seed'], d['peak_gb'], d['secs']))
    print('TERTINGGI: %.2f GB pada %s/%s' % (rows[0]['peak_gb'], rows[0]['phase'],
                                             rows[0]['tag']))

class A: pass

def build(**over):
    """Argumen driver. Setiap atribut diberi nilai di sini, jadi penjaga di bawah
    menangkap salah ketik nama alih-alih membiarkannya lewat sebagai default."""
    x = A()
    x.scores = x.labels = None
    x.eval_scores = x.eval_labels = None
    x.max_rows = None
    x.dataset = 'unset'
    x.reports_dir = 'pcc/reports'
    x.alpha, x.n_cal = 0.10, 25
    x.heldout_frac = 0.30
    x.frac_desc, x.frac_cal = 0.0, 0.70
    x.phi = 'head'
    x.head_weights = x.head_bias = None
    x.distance_holdout = 'w_cos_knn_1'
    x.stat = 'worst'
    x.ccc_root = LTC_DIR if os.path.isdir(LTC_DIR) else None
    x.competitors = False
    x.score = 'thr'
    x.cal_depth = None
    x.eval_depth = None
    x.min_eval_rows = None
    x.lam_override = None
    x.n_star_rule = 'oos'
    x.no_recalibrate = False
    x.feature_group = 'all'
    # ditambahkan untuk addenda ulasan. Penjaga di bawah menolak atribut yang tidak
    # didaftarkan di sini, jadi melewatkan satu baris ini = 20 run gagal di menit pertama.
    x.frac_recal = 0.0
    x.dist_metric = 'cosine'
    x.knn_ks = None
    x.drop_features = ()
    x.seed = 0
    x.name = None
    x.print_json = False
    for k, v in over.items():
        assert hasattr(x, k), 'atribut tak dikenal: ' + k   # salah ketik = diam
        setattr(x, k, v)
    assert x.scores and x.labels, 'scores/labels wajib'
    if x.scores == PRIMARY_S and 'max_rows' not in over:
        raise AssertionError(
            'max_rows harus DILEWATKAN eksplisit pada dump primer. Tanpa itu dump penuh '
            '(1.153.051 baris, 4,61 GB) dimuat, sesi crash, dan angkanya memakai 524 '
            'baris kalibrasi per kelas bukan 177 -- tidak sebanding dengan tabel mana pun '
            'di paper. Pakai max_rows=MAX_ROWS, atau nilai lain kalau itu memang sumbunya.')
    return x

class _Stale(Exception):
    """Cache yang sah tapi ditulis di config lain -- bukan kerusakan berkas."""

def report_name(phase, tag, seed):
    return 'nb13_{}_{}_s{}'.format(phase, tag, seed).replace('.', 'p')

def is_cached(phase, tag, seed):
    """Sudah ada di cache Drive? Dipakai untuk melewati PERSIAPAN, bukan cuma run.

    Fase 4 mengubah 90 berkas .pt jadi .npy SEBELUM memanggil run_one. Tanpa
    pemeriksaan ini, run yang dilanjutkan tetap menulis ~13 GB .npy lalu membuang
    hasilnya karena laporannya sudah tercache -- puluhan menit untuk apa-apa."""
    p = os.path.join(CACHE_DIR, report_name(phase, tag, seed) + '.json')
    return os.path.exists(p) and os.path.getsize(p) > 200

def run_one(phase, tag, **over):
    """Satu konfigurasi, dan ia DILEWATI kalau laporannya sudah ada di Drive.

    Notebook 10 punya resume, notebook 12 tidak -- dan itu kesalahan yang mahal: satu
    kernel restart di fase 3 membuang seluruh fase 1 dan 2 juga. CACHE_DIR sengaja
    TIDAK berstempel waktu, supaya run berikutnya benar-benar menemukannya."""
    t0 = time.time()
    nm = report_name(phase, tag, over.get('seed', 0))
    cached = os.path.join(CACHE_DIR, nm + '.json')
    if os.path.exists(cached) and os.path.getsize(cached) > 200:
        try:
            pay = json.load(open(cached))
            _stale = False
            # THE cache key is (phase, tag, seed) and NOT the config, so a report
            # written before a config change would be reused silently under the new
            # settings. Phase 3 hit exactly that: its seed 0 was cached at the old
            # MAX_ROWS_COMP and returned alongside a fresh seed 1 computed at the new
            # one. The stored config is therefore compared against the config this
            # call would build, on every field that can move a number.
            _want = vars(build(**over))
            _got = pay.get('config', {})
            # JSON has no tuples. `drop_features` defaults to () and comes back as [],
            # `knn_ks` goes (1,5,10,50) -> [1,5,10,50], so a raw str() comparison marks
            # EVERY run stale and resume recomputes the whole notebook. Found by
            # test_notebook_resume.py before it cost a second session.
            def _norm(v):
                return list(v) if isinstance(v, (list, tuple)) else v
            _diff = [k for k in ('scores', 'labels', 'eval_scores', 'eval_labels',
                                 'max_rows', 'alpha', 'n_cal', 'heldout_frac',
                                 'frac_desc', 'frac_cal', 'phi', 'score', 'stat',
                                 'cal_depth', 'lam_override', 'n_star_rule',
                                 'no_recalibrate', 'feature_group', 'competitors',
                                 'head_weights', 'seed',
                                 'frac_recal', 'dist_metric', 'knn_ks', 'drop_features')
                     if str(_norm(_got.get(k))) != str(_norm(_want.get(k)))]
            if _diff:
                print('  {:34s} cache BASI (beda: {}) -- dihitung ulang'.format(
                    tag, ', '.join(_diff[:4])), flush=True)
                _stale = True
                pay = None
            if _stale:
                raise _Stale()
            RESULTS.append(dict(phase=phase, tag=tag, seed=over.get('seed', 0),
                                secs=pay.get('runtime_seconds') or 0.0,
                                res=pay['results'], conclusion=pay['conclusion']))
            shutil.copy2(cached, os.path.join('pcc/reports', nm + '.json'))
            t2 = pay['results'].get('table_2_heldout')
            st = t2['primary_stat'] if t2 else None
            print('  {:34s} s{} CACHE  | T2 {:+.4f} | {}'.format(
                tag, over.get('seed', 0),
                t2['delta'].get(st, float('nan')) if t2 else float('nan'),
                pay['conclusion']), flush=True)
            return pay['results']
        except _Stale:
            pass                     # sudah dilaporkan di atas, bukan kerusakan
        except Exception as e:
            print('  cache RUSAK, dihitung ulang:', nm, e)
    try:
        x = build(**over)
        _stop, _pk = threading.Event(), [rss()]
        _th = threading.Thread(target=_sample_peak, args=(_stop, _pk), daemon=True)
        _th.start()
        try:
            r = drv.run(x)
        finally:
            _stop.set()
            _th.join(timeout=2)
        PEAKS.append(dict(phase=phase, tag=tag, seed=over.get('seed', 0),
                          peak_gb=_pk[0], secs=time.time() - t0))
        c = drv.verdict(r, x.stat)
        pth = write_report(x.reports_dir, nm, hypothesis=drv.HYPOTHESIS,
                           pass_criteria=drv.PASS_CRITERIA, config=vars(x),
                           seed=x.seed, results=r, conclusion=c, started_at=t0)
        # ke cache Drive SEGERA, bukan di akhir fase: yang mahal di sini adalah
        # satu run, dan satu run yang hilang tidak boleh menarik yang lain
        try:
            shutil.copy2(str(pth), os.path.join(CACHE_DIR, nm + '.json'))
        except Exception as e:
            print('   gagal menulis cache:', e)
        RESULTS.append(dict(phase=phase, tag=tag, seed=x.seed,
                           secs=time.time() - t0, res=r, conclusion=c))
        t2 = r.get('table_2_heldout')
        st = t2['primary_stat'] if t2 else None
        print('  {:34s} s{} {:5.0f}s | puncak {:5.2f} GB | lam {:.3f} | T2 {:+.4f} | {}'
              .format(tag, x.seed, time.time() - t0, _pk[0], r['pcc']['lambda'],
                      t2['delta'].get(st, float('nan')) if t2 else float('nan'), c),
              flush=True)
        del r
        gc.collect()
        return RESULTS[-1]['res']
    except Exception as e:
        FAILED.append(dict(phase=phase, tag=tag, error=type(e).__name__ + ': ' + str(e)))
        print('  {:34s} GAGAL: {}'.format(tag, str(e)[:140]), flush=True)
        traceback.print_exc()
        return None

def n_cal_for(cal_rows_per_class, wanted=(25, 15, 10, 5)):
    """n_cal terbesar yang MASIH tercapai pada irisan sedalam ini.

    n_cal adalah kriteria pra-registrasi, jadi driver menolak menurunkannya sendiri.
    Menurunkannya di sini eksplisit, dan nilai yang dibuang dicatat -- bukan diam-diam
    dipilih supaya run-nya lolos."""
    ok = [n for n in wanted if n <= cal_rows_per_class]
    return (ok[0] if ok else 1), [n for n in wanted if n > cal_rows_per_class]

print('runner siap | pesaing dari', LTC_DIR if os.path.isdir(LTC_DIR) else 'TIDAK ADA')

## Status cache — apa yang akan dipakai ulang, apa yang dihitung ulang

Jalankan ini **sebelum** fase apa pun. Ia membaca cache Drive dan, untuk tiap laporan yang
sudah ada, membandingkan config tersimpannya dengan config yang akan dibangun notebook ini
sekarang. Tidak ada yang dihitung; ini murni laporan.

Sesi yang mati tetap meninggalkan jejaknya: `run_one` menulis ke cache **segera** setelah
tiap run selesai, bukan di akhir fase. Yang rusak sebelumnya bukan penulisannya, melainkan
pembandingnya — JSON tidak punya tuple, jadi `drop_features=()` kembali sebagai `[]` dan
semua entri ditandai basi. Itu sudah diperbaiki, jadi entri lama yang **config-nya masih
benar** kini benar-benar dipakai ulang.

In [ ]:
import glob as _g
_files = sorted(_g.glob(CACHE_DIR + '/nb13_*.json'))
print('cache:', CACHE_DIR)
print('laporan tersimpan:', len(_files))
if not _files:
    print('  kosong -- semuanya akan dihitung dari awal')
else:
    _FIELDS = ('scores', 'labels', 'max_rows', 'alpha', 'n_cal', 'heldout_frac',
               'frac_desc', 'frac_cal', 'phi', 'score', 'stat', 'cal_depth',
               'lam_override', 'n_star_rule', 'no_recalibrate', 'feature_group',
               'competitors', 'head_weights', 'seed',
               'frac_recal', 'dist_metric', 'knn_ks', 'drop_features')
    def _n(v):
        return list(v) if isinstance(v, (list, tuple)) else v
    # config yang AKAN dibangun tiap fase, sama persis dengan pemanggilan di bawah
    _common = dict(scores=PRIMARY_S, labels=PRIMARY_Y, dataset=PRIMARY, alpha=ALPHA,
                   n_cal=N_CAL_MAIN, head_weights=HEAD_W, head_bias=HEAD_B)
    _want = {}
    for _s in SEEDS_SPLIT:
        for _tag, _fr in ARMS_R:
            _want[('R', _tag, _s)] = dict(_common, frac_cal=FRAC_CAL_MAIN,
                                          max_rows=MAX_ROWS, frac_recal=_fr, seed=_s)
    for _s in SEEDS_COMP:
        _want[('K', 'competitors', _s)] = dict(_common, frac_cal=FRAC_CAL_COMP,
                                               max_rows=MAX_ROWS_COMP,
                                               competitors=True, seed=_s)
    for _s in SEEDS_MAIN:
        _want[('P', 'pas', _s)] = dict(_common, frac_cal=FRAC_CAL_MAIN,
                                       max_rows=MAX_ROWS, score='pas', seed=_s)
        for _tag, _ov in DESC_ARMS:
            _hold = 'w_cos_knn_{}'.format(min(_ov['knn_ks']) if 'knn_ks' in _ov else 1)
            _want[('D', _tag, _s)] = dict(_common, frac_cal=FRAC_CAL_MAIN,
                                          max_rows=MAX_ROWS, distance_holdout=_hold,
                                          seed=_s, **_ov)
        for _b in ROW_BUDGETS:
            _want[('M', 'rows{}'.format(_b or 'full'), _s)] = dict(
                _common, frac_cal=FRAC_CAL_MAIN, max_rows=_b, seed=_s)

    _reuse, _stale, _orphan = [], [], []
    for _f in _files:
        _pay = json.load(open(_f))
        _got = _pay.get('config', {})
        _nm = os.path.basename(_f)[len('nb13_'):-len('.json')]
        _ph = _nm.split('_')[0]
        _sd = int(_nm.rsplit('_s', 1)[1])
        _tg = _nm[len(_ph) + 1:_nm.rindex('_s')]
        _key = (_ph, _tg, _sd)
        if _key not in _want:
            _orphan.append(_nm); continue
        _w = vars(build(**_want[_key]))
        _d = [k for k in _FIELDS if str(_n(_got.get(k))) != str(_n(_w.get(k)))]
        (_reuse if not _d else _stale).append((_nm, _d))
    print()
    print('DIPAKAI ULANG  :', len(_reuse))
    for _nm, _ in _reuse:
        print('   ', _nm)
    print('DIHITUNG ULANG :', len(_stale), '(config berubah)')
    for _nm, _d in _stale:
        print('   {:34s} beda: {}'.format(_nm, ', '.join(_d[:4])))
    print('YATIM          :', len(_orphan), '(tag tidak ada lagi di notebook ini)')
    for _nm in _orphan:
        print('   ', _nm)
    print()
    print('akan dihitung: {} dari {} run'.format(
        len(_want) - len(_reuse), len(_want)))

## FASE R — PCC-split: berapa harga premis Proposisi 1

Dua lengan berpasangan pada sepuluh split yang sama. `reuse` adalah apa yang dipakai
seluruh paper: `g_θ`, `λ`, dan `c` membaca baris kalibrasi yang **sama**, sehingga premis
Proposisi 1 tidak terpenuhi. `split` menyisihkan 35% baris tiap kelas yang tugasnya
**hanya** memberi kuantil konformal untuk `c`.

Yang dibaca bukan worst-class saja, tapi **coverage marginal pada threshold apa adanya**
(`raw_unmatched`) terhadap target 1−α. Di situlah validitas hidup, dan di situlah pemakaian
ulang baris membayar harganya.

In [ ]:
print('FASE R: PCC-split |', len(SEEDS_SPLIT), 'split | fraksi', FRAC_RECALS)
for s in SEEDS_SPLIT:
    for tag, fr in ARMS_R:
        run_one('R', tag, scores=PRIMARY_S, labels=PRIMARY_Y, dataset=PRIMARY,
                alpha=ALPHA, n_cal=N_CAL_MAIN, frac_cal=FRAC_CAL_MAIN,
                max_rows=MAX_ROWS, head_weights=HEAD_W, head_bias=HEAD_B,
                frac_recal=fr, seed=s)
    mem('setelah split seed {}'.format(s))

## FASE K — pesaing terbit, kini dengan INTERP-Q

Konfigurasi identik dengan fase 3 notebook 12, jadi baris barunya bisa ditempel ke
`tab:competitors` tanpa mencampur statistik. Yang berubah hanya: `interp_q` sekarang ikut
dihitung, dengan τ ∈ {0, 0.5, 0.9, 0.99, 0.999, 1} dan **yang terbaik disimpan** — sama
seperti bandwidth fuzzy, jadi keberpihakannya ke pesaing.

INTERP-Q penting karena ia satu dari sedikit metode terbit yang **terdefinisi** di
`n_y = 0`: kuantil classwise yang tak terdefinisi diganti 1 lebih dulu, lalu dicampur.
Jadi angkanya adalah pengukuran metodenya, bukan pengukuran kebijakan fallback kita.

> **Fase paling berat.** Ia yang membuat sesi notebook 12 mati dua kali, dan penyebabnya
> `compute_rc3p_params`: ia menyalin softmax, lalu `compute_ranks` menyalinnya lagi, lalu
> mengalokasi argsort int64 dan `np.empty((n,K), int)` — empat matriks seukuran penuh di
> dalam satu panggilan, dan puncaknya tidak terlihat oleh pembacaan RSS sebelum/sesudah.
> Konfigurasinya **tidak** dikecilkan untuk menghindari itu: 100k baris dengan
> `frac_cal=0.50` adalah yang dipakai `tab:competitors`, dan mengubahnya membuat baris
> INTERP-Q tidak sebanding dengan enam baris lain di tabel yang sama.
>
> Kalau sesi mati di sini, **restart kernel lalu jalankan sel ini saja**. Resume sudah
> diuji (`pcc/tests/test_notebook_resume.py`), jadi fase R tidak akan dihitung ulang.

In [ ]:
print('FASE K: pesaing + INTERP-Q |', len(SEEDS_COMP), 'split')
print('  biaya terukur ~8 mnt/run pada', MAX_ROWS_COMP, 'baris')
for s in SEEDS_COMP:
    run_one('K', 'competitors', scores=PRIMARY_S, labels=PRIMARY_Y, dataset=PRIMARY,
            alpha=ALPHA, n_cal=N_CAL_MAIN, frac_cal=FRAC_CAL_COMP,
            max_rows=MAX_ROWS_COMP, head_weights=HEAD_W, head_bias=HEAD_B,
            competitors=True, seed=s)
    mem('setelah pesaing seed {}'.format(s))

## FASE P — PAS sebagai sumbu skor

`s_PAS(x,y) = −p̂(y|x)/p̂(y)`, ditranskripsi dari `compute_PAS_scores` di `example.ipynb`
penulisnya. Priornya dari hitungan label **training**, yang ada untuk kelas tanpa baris
kalibrasi — itulah sebabnya PAS terdefinisi di regime kita.

Baris ini menjawab pertanyaan ulasan secara langsung: kalau seseorang sudah memakai PAS,
apakah PCC masih menambah sesuatu? Bandingkan dengan baris LAC/APS/RAPS/SAPS di
`tab:depth`.

In [ ]:
print('FASE P: skor PAS |', len(SEEDS_MAIN), 'split')
# max_rows=MAX_ROWS WAJIB: baris ini masuk tab:depth bersama LAC/APS/RAPS/SAPS, yang
# semuanya berjalan pada 250_000 baris dengan 177 cal/kelas. Di dump penuh PAS mendapat
# ~524 cal/kelas, dan blok kedalaman-kalibrasi di tabel yang sama menunjukkan 100->175
# saja sudah membeli +0.019 -- jadi PAS akan terlihat bagus karena kedalamannya.
for s in SEEDS_MAIN:
    run_one('P', 'pas', scores=PRIMARY_S, labels=PRIMARY_Y, dataset=PRIMARY,
            alpha=ALPHA, n_cal=N_CAL_MAIN, frac_cal=FRAC_CAL_MAIN,
            max_rows=MAX_ROWS, head_weights=HEAD_W, head_bias=HEAD_B,
            score='pas', seed=s)
mem('setelah PAS')

## FASE D — pilihan deskriptor

Ulasan menanyakan seberapa sensitif hasilnya terhadap konstruksi φ. Empat lengan, masing
lima split, semuanya di konfigurasi utama:

- **euclidean** — jarak Euclidean menggantikan cosine. Nama kolomnya sengaja **sama**,
  jadi kedua run bisa dibandingkan baris per baris.
- **nobias** — `w_bias` dibuang. Ini fitur yang paling dekat dengan "label-free": bias
  adalah parameter model, jadi ini bukan soal label, tapi soal apakah ia terpakai.
- **knn_1_2_5** dan **knn_1_10_20_100** — jumlah tetangga lain. Yang default (1,5,10,50)
  ditetapkan sebelum run pertama dan tidak pernah disetel; ini yang membuktikannya.

Catatan: `w_cos_knn_{k terkecil}` selalu ditahan sebagai `distance_holdout`, jadi tiap
lengan menahan fiturnya sendiri dan tidak ada lengan yang diuntungkan.

In [ ]:
print('FASE D: pilihan deskriptor |', len(DESC_ARMS), 'lengan x', len(SEEDS_MAIN), 'split')
for tag, over in DESC_ARMS:
    hold = 'w_cos_knn_{}'.format(min(over['knn_ks']) if 'knn_ks' in over else 1)
    for s in SEEDS_MAIN:
        run_one('D', tag, scores=PRIMARY_S, labels=PRIMARY_Y, dataset=PRIMARY,
                alpha=ALPHA, n_cal=N_CAL_MAIN, frac_cal=FRAC_CAL_MAIN,
                max_rows=MAX_ROWS, head_weights=HEAD_W, head_bias=HEAD_B,
                distance_holdout=hold, seed=s, **over)
    mem('setelah lengan {}'.format(tag))

## FASE M — apakah kesimpulannya bergantung pada anggaran baris?

Setiap tabel di paper berjalan pada `max_rows=250_000`, dan asalnya **anggaran memori**: dump
CCC penuh 1.153.051 × 1.000 float32 = 4,61 GB, dan memuatnya penuh lalu menyalin turunannya
melewati RAM Colab. Itu tercatat di sel 2 notebook 12 sejak awal.

Subsample-nya sah — ia **fraksi acak per kelas**, jadi exchangeability utuh dan jaminan
konformalnya tidak tersentuh, dan kriteria pra-registrasi ≥84 sampel/kelas terpenuhi dengan
margin 2,7×. Tapi "sah" itu argumen. Fase ini menjadikannya pengukuran, dan **ia bisa gagal**.

Prediksinya, dari `tab:depth` dan `fig:evaldepth`: efeknya **naik** dengan anggaran baris,
karena kedalaman kalibrasi dan evaluasi keduanya naik. Kalau itu benar, 250k **konservatif** —
angka yang dilaporkan lebih kecil daripada yang bisa dicapai, bukan lebih besar. Kalau efeknya
justru **turun** dengan anggaran, maka angka utama bergantung pada subsample dan itu harus
masuk Limitations.

Dump penuh tidak ada di sini karena puncaknya ~9,2 GB — itulah yang membuat sesi crash.
Tambahkan `None` ke `ROW_BUDGETS` kalau kau pindah ke runtime dengan RAM lebih besar.

In [ ]:
print('FASE M: anggaran baris |', ROW_BUDGETS, '|', len(SEEDS_MAIN), 'split')
for b in ROW_BUDGETS:
    pk, f_pc, e_pc = footprint(b, FRAC_CAL_MAIN)
    print('  {:>9} baris -> {:.2f} GB puncak, {} cal/kelas, {} eval/kelas'.format(
        b or 'PENUH', pk, f_pc, e_pc))
    for s in SEEDS_MAIN:
        run_one('M', 'rows{}'.format(b or 'full'),
                scores=PRIMARY_S, labels=PRIMARY_Y, dataset=PRIMARY,
                alpha=ALPHA, n_cal=N_CAL_MAIN, frac_cal=FRAC_CAL_MAIN,
                max_rows=b, head_weights=HEAD_W, head_bias=HEAD_B, seed=s)
    mem('setelah anggaran {}'.format(b))

## Ringkasan — tepat angka yang dibutuhkan dua TODO apendiks

Sel ini mencetak apa yang masuk `app:split` dan `app:descriptors`, plus baris INTERP-Q untuk
`tab:competitors` dan baris PAS untuk `tab:depth`. Tempel keluarannya; jangan salin ulang
dengan tangan.

In [ ]:
import numpy as np, collections
def rows(phase, tag=None):
    return [r for r in RESULTS if r['phase'] == phase and (tag is None or r['tag'] == tag)]

def ci(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < 2:
        return (float(v.mean()) if len(v) else float('nan'), float('nan'), float('nan'))
    m, h = v.mean(), 1.96 * v.std(ddof=1) / np.sqrt(len(v))
    return float(m), float(m - h), float(m + h)

def t2(r):
    return r['res'].get('table_2_heldout') or {}

print('=' * 78)
print('APP:SPLIT  -- premis Proposisi 1')
print('=' * 78)
for tag, _fr in ARMS_R:
    R = rows('R', tag)
    if not R:
        print('  ', tag, 'TIDAK ADA'); continue
    st = t2(R[0]).get('primary_stat', 'worst')
    ru = [t2(r).get('raw_unmatched', {}) for r in R]
    cov  = ci([u.get('pcc', {}).get('marginal_cov') for u in ru])
    base = ci([u.get('uncorrected', {}).get('marginal_cov') for u in ru])
    size = ci([u.get('pcc', {}).get('avg_set_size') for u in ru])
    dw   = ci([t2(r)['delta'][st] for r in R])
    holds = {r['res']['pcc']['provenance'].get('prop1_premise_holds') for r in R}
    nrec  = {r['res'].get('recal_rows') for r in R}
    print('  {:6s} n={:2d} premis_terpenuhi={} baris_recal={}'.format(
        tag, len(R), holds, sorted(nrec)[:3]))
    print('        coverage marginal PCC   {:.4f} [{:.4f},{:.4f}]  (target {:.3f})'.format(
        cov[0], cov[1], cov[2], 1 - ALPHA))
    print('        coverage marginal dasar {:.4f} [{:.4f},{:.4f}]'.format(*base))
    print('        defisit vs dasar        {:+.4f}'.format(cov[0] - base[0]))
    print('        ukuran set PCC (mentah) {:.3f}'.format(size[0]))
    print('        delta worst-class       {:+.4f} [{:+.4f},{:+.4f}]'.format(*dw))
for tag, fr in ARMS_R[1:]:
    if not (rows('R', 'reuse') and rows('R', tag)):
        continue
    a = {r['seed']: r for r in rows('R', 'reuse')}
    b = {r['seed']: r for r in rows('R', tag)}
    sh = sorted(set(a) & set(b))
    st = t2(a[sh[0]]).get('primary_stat', 'worst')
    d = [t2(b[s])['delta'][st] - t2(a[s])['delta'][st] for s in sh]
    print('  BERPASANGAN {} vs reuse, {} split:'.format(tag, len(sh)))
    print('     delta worst-class   {:+.4f} [{:+.4f},{:+.4f}]'.format(*ci(d)))
    dc = [t2(b[s]).get('raw_unmatched', {}).get('pcc', {}).get('marginal_cov', np.nan)
          - t2(a[s]).get('raw_unmatched', {}).get('pcc', {}).get('marginal_cov', np.nan)
          for s in sh]
    print('     coverage marginal   {:+.5f} [{:+.5f},{:+.5f}]'.format(*ci(dc)))

print()
print('=' * 78)
print('ANGGARAN BARIS -- apakah kesimpulannya bergantung pada subsample?')
print('=' * 78)
print('  nb12 dan seluruh paper memakai 250_000. Kalau delta NAIK dengan anggaran, maka')
print('  250_000 konservatif. Kalau TURUN, angka utama bergantung pada subsample.')
print()
print('  {:>10s} {:>9s} {:>20s} {:>9s} {:>8s} {:>9s}'.format(
    'baris', 'd_worst', '95% CI', 'plafon', 'cal/kls', 'eval/kls'))
_M = []
for b in ROW_BUDGETS:
    R = rows('M', 'rows{}'.format(b or 'full'))
    if not R:
        print('  {:>10s} TIDAK ADA'.format(str(b or 'PENUH'))); continue
    st = t2(R[0])['primary_stat']
    m = ci([t2(r)['delta'][st] for r in R])
    c = ci([t2(r)['delta_oracle'][st] for r in R])
    calpc = np.mean([r['res']['cal_rows_per_seen_class']['median'] for r in R])
    evpc = t2(R[0])['measurability']['median_eval_per_class']
    _M.append((b or 1_153_051, m[0], m[1], m[2]))
    print('  {:>10s} {:+9.4f} [{:+8.4f},{:+8.4f}] {:+9.4f} {:9.0f} {:8.0f}'.format(
        str(b or 'PENUH'), m[0], m[1], m[2], c[0], calpc, evpc))
if len(_M) >= 2:
    xs = np.array([x for x, _, _, _ in _M], float)
    ys = np.array([y for _, y, _, _ in _M], float)
    slope = np.polyfit(np.log(xs), ys, 1)[0]
    print()
    print('  kemiringan d_worst terhadap log(baris): {:+.4f} per e-fold'.format(slope))
    # Sebuah kemiringan bukan bukti arah. Ujinya: apakah tiap estimasi titik jatuh di
    # DALAM interval kedua yang lain? Kalau ya, lima split tidak bisa memisahkan ketiga
    # anggaran, dan menyebutnya "naik" adalah klaim yang tidak ditanggung datanya --
    # kesalahan yang versi pertama sel ini lakukan pada 2026-08-19.
    _sep = False
    for i, (_, mi, loi, hii) in enumerate(_M):
        for j, (_, mj, loj, hij) in enumerate(_M):
            if i != j and not (loj <= mi <= hij):
                _sep = True
    if not _sep:
        print('  TIDAK TERPISAHKAN -> tiap estimasi titik jatuh di dalam interval kedua')
        print('  yang lain, jadi lima split tidak bisa membedakan ketiga anggaran. Yang')
        print('  boleh diklaim: anggaran baris TIDAK menentukan kesimpulan pada rentang ini.')
        print('  Itu jawaban yang cukup untuk pertanyaan "apakah subsample mengarang hasil".')
    elif slope > 0:
        print('  NAIK dan terpisahkan -> 250_000 konservatif; angka yang dilaporkan lebih')
        print('  KECIL daripada yang bisa dicapai.')
    else:
        print('  TURUN dan terpisahkan -> angka utama bergantung pada subsample. HARUS masuk')
        print('  Limitations, dan tabel utama dijalankan ulang pada anggaran terbesar yang muat.')

print()
print('=' * 78)
print('TAB:COMPETITORS -- baris INTERP-Q (dan seluruh tabel, ulang)')
print('=' * 78)
K = rows('K', 'competitors')
if K:
    st = t2(K[0])['primary_stat']
    comp = collections.defaultdict(list)
    for r in K:
        for nm, e in (t2(r).get('delta_competitors') or {}).items():
            comp[nm].append(e.get(st, np.nan))
    pcc = ci([t2(r)['delta'][st] for r in K])
    cel = ci([t2(r)['delta_oracle'][st] for r in K])
    und = collections.defaultdict(list)
    for r in K:
        for nm, e in (t2(r).get('competitors') or {}).items():
            und[nm].append(e.get('n_classes_undefined', 0))
    hp = collections.defaultdict(list)
    for r in K:
        for nm, e in (t2(r).get('competitors') or {}).items():
            hp[nm].append(str(e.get('hyperparameter', '-')))
    print('  {:28s} {:>9s} {:>10s}  {}'.format('metode', 'd_worst', 'takdefinisi', 'hp'))
    for nm in sorted(comp):
        m = ci(comp[nm])
        print('  {:28s} {:+9.4f} {:>10s}  {}'.format(
            nm, m[0], '{}/{}'.format(int(np.mean(und[nm])), t2(K[0])['n_classes']),
            collections.Counter(hp[nm]).most_common(1)[0][0]))
    print('  {:28s} {:+9.4f}  <-- kita'.format('PCC', pcc[0]))
    print('  {:28s} {:+9.4f}  ({:.0f}% diambil PCC)'.format(
        'plafon oracle', cel[0], 100 * pcc[0] / cel[0] if cel[0] > 0 else float('nan')))
    print('  ukuran set target:', round(t2(K[0])['target_avg_set_size'], 4))
else:
    print('  TIDAK ADA')

print()
print('=' * 78)
print('TAB:DEPTH -- baris PAS')
print('=' * 78)
P = rows('P', 'pas')
if P:
    st = t2(P[0])['primary_stat']
    m = ci([t2(r)['delta'][st] for r in P])
    c = ci([t2(r)['delta_oracle'][st] for r in P])
    print('  PAS  d_worst {:+.4f} [{:+.4f},{:+.4f}] | plafon {:+.4f} | {:.0f}% plafon'
          .format(m[0], m[1], m[2], c[0], 100 * m[0] / c[0] if c[0] > 0 else float('nan')))
    print('  regime:', t2(P[0])['measurability']['regime'],
          '| lambda:', [r['res']['pcc']['lambda'] for r in P])
else:
    print('  TIDAK ADA')

print()
print('=' * 78)
print('APP:DESCRIPTORS -- sensitivitas')
print('=' * 78)
print('  {:22s} {:>9s} {:>20s} {:>8s} {:>7s}'.format(
    'lengan', 'd_worst', '95% CI', 'lambda', 'mse_g'))
base = rows('R', 'reuse') or []
if base:
    st = t2(base[0]).get('primary_stat', 'worst')
    m = ci([t2(r)['delta'][st] for r in base])
    ms = np.mean([r['res']['pcc'].get('n_star_mse_crossing_secondary', {})
                  .get('gtheta_mse', np.nan) for r in base])
    print('  {:22s} {:+9.4f} [{:+7.4f},{:+7.4f}] {:>8s} {:7.4f}'.format(
        'default (cosine)', m[0], m[1], m[2],
        str(sorted({r['res']['pcc']['lambda'] for r in base})), ms))
for tag, _ in DESC_ARMS:
    D = rows('D', tag)
    if not D:
        print('  {:22s} TIDAK ADA'.format(tag)); continue
    st = t2(D[0])['primary_stat']
    m = ci([t2(r)['delta'][st] for r in D])
    ms = np.mean([r['res']['pcc'].get('n_star_mse_crossing_secondary', {})
                  .get('gtheta_mse', np.nan) for r in D])
    print('  {:22s} {:+9.4f} [{:+7.4f},{:+7.4f}] {:>8s} {:7.4f}'.format(
        tag, m[0], m[1], m[2],
        str(sorted({r['res']['pcc']['lambda'] for r in D})), ms))
    print('  {:22s} fitur: {}'.format('', D[0]['res']['pcc']['features']))

print()
print('=' * 78)
print('PUNCAK RSS -- diambil SELAMA tiap run, bukan sebelum/sesudah')
print('=' * 78)
peaks_report()

print()
print('GAGAL:', len(FAILED))
for f in FAILED:
    print('  ', f)